# Meta API Discovery and Analysis (Pure Groggy)
## Dynamic Graph Generation via Groggy Generator

This notebook demonstrates the **Meta API Discovery System** using **only Groggy**.
We dynamically build the meta-graph, then load it via the **Groggy generator** to analyze Groggy’s own API.

**Key goals:**
- **Dynamic build**: regenerate the meta-graph from live Groggy objects
- **Pure Groggy**: no pandas, no external graph libs
- **Graph-as-Data**: treat Groggy’s API as a dataset


## 1. Setup
Import Groggy and prepare paths.


In [ ]:
from pathlib import Path
import groggy as gr

print("✅ Groggy imported")
print("Groggy version:", getattr(gr, "__version__", "unknown"))


## 2. Build the Meta-Graph (Dynamic)
Run the dynamic builder to regenerate the bundle.


In [ ]:
from notes.meta_api_discovery.meta_api_graph_builder import MetaAPIGraphBuilder

output_dir = Path("notes/meta_api_discovery/meta_api_graph_bundle")

builder = MetaAPIGraphBuilder(run_tests=True)
builder.discover()
builder.build_meta_graph()
summary = builder.export_bundle(output_dir)

print("Meta API Graph built")
print("Nodes:", summary["meta_graph_stats"]["nodes"])
print("Edges:", summary["meta_graph_stats"]["edges"])
print("Types discovered:", summary["meta_graph_stats"]["types_discovered"])


## 3. Load the Graph via Generator
Use Groggy’s generator to load the freshly built meta-graph.


In [ ]:
api_graph = gr.generators.meta_api_graph()

graph_table = api_graph.table()
nodes_table = graph_table.nodes()
edges_table = graph_table.edges()

print("Graph loaded")
print("Nodes:", api_graph.node_count())
print("Edges:", api_graph.edge_count())


## 4. Method Count Snapshot
Analyze method counts per object using Groggy tables/arrays only.


In [ ]:
method_counts = nodes_table.column("methods_count")
print("Method count stats:")
print(method_counts.describe())

# Show top objects by method count
sorted_nodes = nodes_table.sort_by("methods_count", ascending=False)
print("
Top objects by method count:")
for i in range(min(10, sorted_nodes.nrows)):
    row = sorted_nodes[i]
    print(f"- {row['type_name']}: {row['methods_count']}")


## 5. Return Type Coverage
How were return types determined?


In [ ]:
sources = edges_table.column("return_type_source")
print("Return type source counts:")
print(sources.value_counts())


## 6. Unknown Return Types
Spot-check methods still marked as Unknown.


In [ ]:
unknown = edges_table.filter(lambda row: row["return_type"] == "Unknown")
print("Unknown return types:", unknown.nrows)
print(unknown.head(10).rich_display())


## 7. Object Type Distribution
Count methods by object type.


In [ ]:
object_types = edges_table.column("object_type")
print(object_types.value_counts())


## 8. Meta-Graph Structure
Basic connectivity and density checks.


In [ ]:
print("Connected:", api_graph.is_connected())
print("Density:", api_graph.density())
print("Node count:", api_graph.node_count())
print("Edge count:", api_graph.edge_count())


## Notes
- This notebook uses **only Groggy** for tables, arrays, and graph operations.
- The meta-graph is dynamically built, then loaded via `gr.generators.meta_api_graph()`.
- No pandas or external graph libraries are used.
